In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
pip install neuraloperator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 8.1 MB/s eta 0:00:00


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Colab-ready Cahn-Hilliard 2D strict benchmark with:
  1) TB-AWFNO : temporal-basis alternating windowed FNO
  2) AWFNO    : alternating windowed FNO
  3) WFNO     : windowed FNO baseline
  4) FNO      : non-windowed full-trajectory FNO baseline
  5) Persistence

Expected .pt file:
    X_input  : (N, Nx, Ny, 1)
    Y_output : (N, T, Nx, Ny, 1)

Internal layout:
    U = (N, Nx, Ny, Nt), with Nt=T+1 and U[...,0]=X_input.

Strict protocol:
    - train/val/test trajectories are disjoint;
    - common test set for all sample sizes;
    - temporal basis is computed from TRAIN windows only;
    - no test windows are constructed;
    - no test temporal-basis coefficients are projected;
    - windowed rollout starts from true u0 only;
    - later windowed inputs are previous predictions;
    - FNO maps u0 directly to the full future trajectory;
    - no Euler reconstruction.

Losses:
    - TB-AWFNO, AWFNO, WFNO:
          local window MSE + FINAL_LOSS_WEIGHT * final snapshot of each window
    - FNO:
          full future trajectory MSE only, no extra final-time term

Metric:
    absolute physical L2, trapezoidal in time, including t=0.

Colab usage:
    1) Upload the .pt file to Google Drive.
    2) Set DATA_FILE if needed:
       os.environ["DATA_FILE"] = "/content/drive/MyDrive/path/to/cahn_hilliard_train_grid_16x16.pt"
    3) Run this script/cell.
"""

import os, gc, json, copy, time, random, hashlib
from pathlib import Path
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def env_bool(name, default=False):
    value = os.environ.get(name, None)
    if value is None:
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}
def env_str(name, default):
    return os.environ.get(name, default)


def env_int(name, default):
    value = os.environ.get(name, None)
    return int(value) if value not in (None, "") else int(default)


def env_float(name, default):
    value = os.environ.get(name, None)
    return float(value) if value not in (None, "") else float(default)


def env_list_int(name, default_list):
    value = os.environ.get(name, None)
    if value is None or str(value).strip() == "":
        return list(default_list)
    return [int(x.strip()) for x in str(value).split(",") if x.strip()]

def env_int_list(name, default):
    raw = os.environ.get(name, None)
    if raw is None:
        return list(default)
    raw = raw.strip()
    if not raw:
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


# ============================================================
# 1. CONFIG
# ============================================================

SEED = env_int("SEED", 1234)


DT_PHYSICAL = env_float("DT_PHYSICAL", 0.01)
X_LEFT, X_RIGHT = 0.0, 1.0
Y_LEFT, Y_RIGHT = 0.0, 1.0

SAMPLE_SIZES = env_list_int("SAMPLE_SIZES", [25, 50, 100, 200, 400])
N_TRAIN_POOL = max(SAMPLE_SIZES)
N_VAL = env_int("N_VAL", 30)
N_TEST = env_int("N_TEST", 70)
SPLIT_SEED = env_int("SPLIT_SEED", 2026)
STRICT_REQUIRE_ENOUGH_SAMPLES = env_bool("STRICT_REQUIRE_ENOUGH_SAMPLES", True)

WINDOW_K = env_int("WINDOW_K", 5)
WINDOW_STRIDE = env_int("WINDOW_STRIDE", WINDOW_K)
N_TIME_MODES = env_int("N_TIME_MODES", 4)

BATCH_SIZE = env_int("BATCH_SIZE", 32)
EPOCHS = env_int("EPOCHS", 130)
LR = env_float("LR", 1e-3)
WEIGHT_DECAY = env_float("WEIGHT_DECAY", 1e-6)

WIDTH = env_int("WIDTH", 32)
N_LAYERS = env_int("N_LAYERS", 4)
MODES_X = env_int("MODES_X", 16)
MODES_Y = env_int("MODES_Y", 16)
LOCAL_RADIUS = env_int("LOCAL_RADIUS", 2)

LOSS_KIND = env_str("LOSS_KIND", "mse")
HUBER_BETA = env_float("HUBER_BETA", 0.02)
FINAL_LOSS_WEIGHT = env_float("FINAL_LOSS_WEIGHT", 1.0)
GRAD_LOSS_WEIGHT = env_float("GRAD_LOSS_WEIGHT", 0.0)
NORMALIZE_LOSS = env_bool("NORMALIZE_LOSS", True)

# For Cahn-Hilliard, mean conservation is useful for windowed rollout.
ENFORCE_MEAN = env_bool("ENFORCE_MEAN", True)

FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)
USE_SPLIT_CACHE = env_bool("USE_SPLIT_CACHE", True)
SAVE_SPLIT_CACHE = env_bool("SAVE_SPLIT_CACHE", True)

RUN_EXACT_HASH_OVERLAP_CHECK = env_bool("RUN_EXACT_HASH_OVERLAP_CHECK", True)
RUN_DUPLICATE_REPORT_INSIDE_SPLITS = env_bool("RUN_DUPLICATE_REPORT_INSIDE_SPLITS", True)
RUN_ROLLOUT_INPUT_GAP_CHECK = env_bool("RUN_ROLLOUT_INPUT_GAP_CHECK", True)
DEBUG_SHUFFLE_TRAIN_OUTPUTS = env_bool("DEBUG_SHUFFLE_TRAIN_OUTPUTS", False)
DEBUG_RANDOM_BASIS = env_bool("DEBUG_RANDOM_BASIS", False)
INPUT_GAP_TOL = env_float("INPUT_GAP_TOL", 1e-12)

SAVE_PLOTS = env_bool("SAVE_PLOTS", True)
MAKE_TRAINING_CURVE_PLOTS = env_bool("MAKE_TRAINING_CURVE_PLOTS", True)
MAKE_L2_PER_TIME_PLOT = env_bool("MAKE_L2_PER_TIME_PLOT", True)
MAKE_FIELD_PLOTS = env_bool("MAKE_FIELD_PLOTS", True)
SHOW_PLOTS = env_bool("SHOW_PLOTS", False)
PLOT_TIME_INDICES_MANUAL_RAW = env_str("PLOT_TIME_INDICES", "5,10,final")
MAX_TEST_CASES_TO_PLOT = env_int("MAX_TEST_CASES_TO_PLOT", 8)
PLOT_DPI = env_int("PLOT_DPI", 140)
L2_CHUNK_SIZE = env_int("L2_CHUNK_SIZE", 16)

# False by default to keep Colab RAM and Drive usage controlled.
SAVE_ROLLOUT_ARRAYS = env_bool("SAVE_ROLLOUT_ARRAYS", False)

PLOT_TIME_INDICES_MANUAL = []
for item in PLOT_TIME_INDICES_MANUAL_RAW.split(","):
    item = item.strip()
    if not item:
        continue
    if item.lower() == "final":
        PLOT_TIME_INDICES_MANUAL.append("final")
    else:
        PLOT_TIME_INDICES_MANUAL.append(int(item))

DATA_FILE = Path("/content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt")
OUTPUT_ROOT = Path("/content/drive/MyDrive/anus/ch2d_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_FILE =", DATA_FILE)
print("exists    =", DATA_FILE.exists())
print("OUTPUT    =", OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_FILE =", DATA_FILE)
print("exists    =", DATA_FILE.exists())
print("OUTPUT    =", OUTPUT_ROOT)

CACHE_ROOT = OUTPUT_ROOT / (
    f"cache_CH2D_TB_AWFNO_AWFNO_WFNO_FNO_EXACT_LKL_"
    f"sizes{'-'.join(map(str, SAMPLE_SIZES))}_"
    f"K{WINDOW_K}_M{N_TIME_MODES}_W{WIDTH}_"
    f"mx{MODES_X}_my{MODES_Y}_R{LOCAL_RADIUS}_"
    f"mean{int(ENFORCE_MEAN)}"
)
if DEBUG_SHUFFLE_TRAIN_OUTPUTS:
    CACHE_ROOT = Path(str(CACHE_ROOT) + "_SHUFFLED_TARGETS")
if DEBUG_RANDOM_BASIS:
    CACHE_ROOT = Path(str(CACHE_ROOT) + "_RANDOM_BASIS")

MODEL_DIR = CACHE_ROOT / "models"
PLOT_DIR = CACHE_ROOT / "plots"
DATA_DIR = CACHE_ROOT / "data"
RESULTS_DIR = CACHE_ROOT / "results"
for d in [MODEL_DIR, PLOT_DIR, DATA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Matplotlib style, Colab-safe.
plt.rcdefaults()
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "legend.fontsize": 10,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 110,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# ============================================================
# 2. REPRODUCIBILITY / DEVICE
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# 3. GLOBAL GEOMETRY
# ============================================================

x_grid_np = None
y_grid_np = None
t_grid = None
Nx = Ny = Nt = None
DX = DY = DT_SNAPSHOT = None
BASE_TAG = "uninitialized"


def set_geometry_from_arrays(U_shape, x_grid, y_grid, dt_physical):
    global x_grid_np, y_grid_np, t_grid, Nx, Ny, Nt, DX, DY, DT_SNAPSHOT
    if len(U_shape) != 4:
        raise ValueError(f"Expected U shape (N,Nx,Ny,Nt). Got {U_shape}")

    _, Nx, Ny, Nt = U_shape
    x_grid_np = np.asarray(x_grid, dtype=np.float32).reshape(-1)
    y_grid_np = np.asarray(y_grid, dtype=np.float32).reshape(-1)

    if len(x_grid_np) != Nx:
        raise ValueError(f"x_grid has len {len(x_grid_np)}, expected Nx={Nx}")
    if len(y_grid_np) != Ny:
        raise ValueError(f"y_grid has len {len(y_grid_np)}, expected Ny={Ny}")

    DX = float(np.mean(np.diff(x_grid_np.astype(np.float64)))) if Nx > 1 else 1.0
    DY = float(np.mean(np.diff(y_grid_np.astype(np.float64)))) if Ny > 1 else 1.0
    DT_SNAPSHOT = float(dt_physical)
    t_grid = np.arange(Nt, dtype=np.float32) * DT_SNAPSHOT

    print("\nGeometry")
    print("--------")
    print("N, Nx, Ny, Nt:", U_shape[0], Nx, Ny, Nt)
    print("x range      :", float(x_grid_np.min()), float(x_grid_np.max()))
    print("y range      :", float(y_grid_np.min()), float(y_grid_np.max()))
    print("t range      :", float(t_grid.min()), float(t_grid.max()))
    print("DX           :", DX)
    print("DY           :", DY)
    print("DT           :", DT_SNAPSHOT)


# ============================================================
# 4. BASIC HELPERS
# ============================================================

def short_hash(obj):
    txt = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.md5(txt.encode("utf-8")).hexdigest()[:10]


def clean_state_dict(sd):
    sd = copy.deepcopy(sd)
    if isinstance(sd, dict) and "_metadata" in sd:
        del sd["_metadata"]
    if hasattr(sd, "_metadata"):
        try:
            del sd._metadata
        except Exception:
            pass
    return OrderedDict((k, v) for k, v in sd.items() if k != "_metadata")


def torch_load_safe(path, map_location=None):
    # PyTorch versions differ in the weights_only argument.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_model_checkpoint(path, model, meta=None):
    torch.save({"state_dict": clean_state_dict(model.state_dict()), "meta": meta or {}}, path)


def load_model_checkpoint(path, model):
    payload = torch_load_safe(path, map_location=device)
    model.load_state_dict(clean_state_dict(payload["state_dict"]), strict=True)
    return payload.get("meta", {})


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def gb(nbytes):
    return nbytes / 1024**3


def print_array_info(name, arr):
    print(
        f"{name:16s}: shape={arr.shape}, dtype={arr.dtype}, "
        f"memory={gb(arr.nbytes):.4f} GB, "
        f"min={float(np.nanmin(arr)):+.6e}, "
        f"max={float(np.nanmax(arr)):+.6e}, "
        f"mean={float(np.nanmean(arr)):+.6e}, "
        f"std={float(np.nanstd(arr)):.6e}"
    )


def time_quadrature_weights(nt):
    w = np.ones(nt, dtype=np.float64)
    if nt > 1:
        w[0] = 0.5
        w[-1] = 0.5
    return w


# ============================================================
# 5. LOAD DATA / SPLIT / HASH CHECKS
# ============================================================

def load_cahn_hilliard_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found:\n{path}\n\n"
            "Set the path before running, for example:\n"
            "os.environ['DATA_FILE'] = '/content/drive/MyDrive/anus/cahn_hilliard_train_grid_16x16.pt'"
        )

    data = torch_load_safe(path, map_location="cpu")

    if "X_input" not in data or "Y_output" not in data:
        raise KeyError("Expected keys 'X_input' and 'Y_output'.")

    X = data["X_input"].float()       # (N,Nx,Ny,1)
    Y = data["Y_output"].float()      # (N,T,Nx,Ny,1)

    if X.ndim != 4 or X.shape[-1] != 1:
        raise ValueError(f"Expected X_input (N,Nx,Ny,1), got {tuple(X.shape)}")
    if Y.ndim != 5 or Y.shape[-1] != 1:
        raise ValueError(f"Expected Y_output (N,T,Nx,Ny,1), got {tuple(Y.shape)}")

    if "x_grid" in data:
        x_grid = data["x_grid"].float().reshape(-1).numpy()
    elif "xg" in data:
        x_grid = data["xg"].float().reshape(-1).numpy()
    else:
        x_grid = np.linspace(X_LEFT, X_RIGHT, X.shape[1], endpoint=False, dtype=np.float32)

    if "y_grid" in data:
        y_grid = data["y_grid"].float().reshape(-1).numpy()
    elif "yg" in data:
        y_grid = data["yg"].float().reshape(-1).numpy()
    else:
        y_grid = np.linspace(Y_LEFT, Y_RIGHT, X.shape[2], endpoint=False, dtype=np.float32)

    X_np = X[..., 0].contiguous().numpy().astype(np.float32)  # (N,Nx,Ny)
    Y_np = Y[..., 0].permute(0, 2, 3, 1).contiguous().numpy().astype(np.float32)  # (N,Nx,Ny,T)
    U = np.concatenate([X_np[..., None], Y_np], axis=-1).astype(np.float32)       # (N,Nx,Ny,T+1)

    if not np.all(np.isfinite(U)):
        raise ValueError("Loaded U contains NaN or Inf.")

    sample_ids = data.get("sample_ids", None)
    if sample_ids is None:
        sample_ids = np.arange(U.shape[0], dtype=np.int64)
    elif torch.is_tensor(sample_ids):
        sample_ids = sample_ids.long().numpy()
    else:
        sample_ids = np.asarray(sample_ids)

    print("\nLoaded Cahn-Hilliard data")
    print("-------------------------")
    print("path    :", path)
    print("X_input :", tuple(X.shape), "->", X_np.shape)
    print("Y_output:", tuple(Y.shape), "->", Y_np.shape)
    print("U       :", U.shape, "= (N,Nx,Ny,Nt)")
    print("sample_ids:", sample_ids.shape)

    return U, x_grid.astype(np.float32), y_grid.astype(np.float32), sample_ids


def split_cache_path():
    data_sig = {
        "DATA_FILE_name": Path(DATA_FILE).name,
        "DATA_FILE_size": Path(DATA_FILE).stat().st_size if Path(DATA_FILE).exists() else None,
        "SAMPLE_SIZES": SAMPLE_SIZES,
        "N_TRAIN_POOL": N_TRAIN_POOL,
        "N_VAL": N_VAL,
        "N_TEST": N_TEST,
        "SPLIT_SEED": SPLIT_SEED,
    }
    tag = short_hash(data_sig)
    return DATA_DIR / f"split_CH2D_{tag}.npz"


def make_split_indices(n_total):
    need = N_TRAIN_POOL + N_VAL + N_TEST
    if need > n_total:
        msg = (
            f"Need {need} trajectories (N_TRAIN_POOL={N_TRAIN_POOL}, "
            f"N_VAL={N_VAL}, N_TEST={N_TEST}), but dataset has only {n_total}. "
            "Reduce SAMPLE_SIZES, N_VAL, or N_TEST."
        )
        if STRICT_REQUIRE_ENOUGH_SAMPLES:
            raise ValueError(msg)
        print("WARNING:", msg)

    rng = np.random.default_rng(SPLIT_SEED)
    perm = rng.permutation(n_total)
    selected = perm[:min(need, n_total)]

    train_idx = selected[:N_TRAIN_POOL]
    val_idx = selected[N_TRAIN_POOL:N_TRAIN_POOL + N_VAL]
    test_idx = selected[N_TRAIN_POOL + N_VAL:N_TRAIN_POOL + N_VAL + N_TEST]

    return train_idx, val_idx, test_idx


def load_split_cache():
    path = split_cache_path()
    if not USE_SPLIT_CACHE or not path.exists():
        return None

    print("\nLoading split cache")
    print("-------------------")
    print(path)

    z = np.load(path, allow_pickle=True)
    return (
        z["U_train_pool"].astype(np.float32),
        z["U_val"].astype(np.float32),
        z["U_test"].astype(np.float32),
        z["train_idx"],
        z["val_idx"],
        z["test_idx"],
        z["x_grid"].astype(np.float32),
        z["y_grid"].astype(np.float32),
    )


def save_split_cache(U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid):
    if not SAVE_SPLIT_CACHE:
        return
    path = split_cache_path()
    print("\nSaving split cache")
    print("------------------")
    print(path)
    np.savez(
        path,
        U_train_pool=U_train_pool,
        U_val=U_val,
        U_test=U_test,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        x_grid=x_grid,
        y_grid=y_grid,
    )
    print("Saved:", path)


def load_or_create_split():
    cached = load_split_cache()
    if cached is not None:
        return cached

    U, x_grid, y_grid, sample_ids = load_cahn_hilliard_file(DATA_FILE)
    train_idx, val_idx, test_idx = make_split_indices(U.shape[0])

    U_train_pool = U[train_idx]
    U_val = U[val_idx]
    U_test = U[test_idx]

    print("\nSplit")
    print("-----")
    print("N total     :", U.shape[0])
    print("N train pool:", len(train_idx))
    print("N val       :", len(val_idx))
    print("N test      :", len(test_idx))
    print("train_idx first/last:", train_idx[:5], train_idx[-5:] if len(train_idx) else [])
    print("val_idx   first/last:", val_idx[:5], val_idx[-5:] if len(val_idx) else [])
    print("test_idx  first/last:", test_idx[:5], test_idx[-5:] if len(test_idx) else [])

    save_split_cache(U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid)

    return U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid


def trajectory_hashes(U):
    U32 = np.ascontiguousarray(U.astype(np.float32, copy=False))
    return [hashlib.sha1(U32[i].tobytes()).hexdigest() for i in range(U32.shape[0])]


def check_exact_overlap(Ua, Ub, name_a, name_b):
    ha = set(trajectory_hashes(Ua))
    hb = set(trajectory_hashes(Ub))
    inter = ha.intersection(hb)
    print(f"\nExact overlap check {name_a} vs {name_b}")
    print("------------------------------------------")
    print("duplicates:", len(inter))
    if inter:
        raise RuntimeError(f"Duplicate trajectories found between {name_a} and {name_b}.")


def report_duplicate_trajectories(U, name):
    hashes = trajectory_hashes(U)
    seen, duplicates = {}, []
    for i, h in enumerate(hashes):
        if h in seen:
            duplicates.append((seen[h], i))
        else:
            seen[h] = i
    print(f"\nDuplicate check inside {name}")
    print("--------------------------------")
    print("total samples :", len(hashes))
    print("unique samples:", len(seen))
    print("duplicates    :", len(duplicates))
    if duplicates:
        print("first duplicate pairs:", duplicates[:10])
    return duplicates


# ============================================================
# 6. L2 METRICS
# ============================================================

def abs_l2_window_batch(pred, true):
    # pred,true: (B,Nx,Ny,K)
    return torch.sqrt(torch.sum((pred - true) ** 2, dim=(1, 2, 3)) * DX * DY * DT_SNAPSHOT + 1e-30)


def abs_l2_field_batch(pred, true):
    # pred,true: (B,Nx,Ny)
    return torch.sqrt(torch.sum((pred - true) ** 2, dim=(1, 2)) * DX * DY + 1e-30)


def l2_per_timestep_np(pred, true, chunk_size=L2_CHUNK_SIZE):
    if pred.shape != true.shape:
        raise ValueError(f"Shape mismatch: pred {pred.shape}, true {true.shape}")
    n, nx, ny, nt = pred.shape
    out = np.zeros((n, nt), dtype=np.float64)
    for i0 in range(0, n, chunk_size):
        i1 = min(i0 + chunk_size, n)
        err = pred[i0:i1].astype(np.float32, copy=False) - true[i0:i1].astype(np.float32, copy=False)
        out[i0:i1] = np.sqrt(np.sum(err * err, axis=(1, 2), dtype=np.float64) * DX * DY + 1e-30)
        del err
        gc.collect()
    return out


def abs_l2_full_traj_np(pred, true):
    l2_t = l2_per_timestep_np(pred, true)
    w = time_quadrature_weights(l2_t.shape[1]).reshape(1, -1)
    return np.sqrt(np.sum(w * l2_t ** 2, axis=1) * DT_SNAPSHOT + 1e-30)


def abs_l2_final_np(pred, true):
    return l2_per_timestep_np(pred, true)[:, -1]


def rollout_l2_metrics(pred, true):
    full = abs_l2_full_traj_np(pred, true)
    final = abs_l2_final_np(pred, true)
    l2_t = l2_per_timestep_np(pred, true)
    return {
        "full_l2_mean": float(np.mean(full)),
        "full_l2_std": float(np.std(full)),
        "full_l2_median": float(np.median(full)),
        "final_l2_mean": float(np.mean(final)),
        "final_l2_std": float(np.std(final)),
        "final_l2_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "all_full_l2": full,
        "all_final_l2": final,
        "all_l2_t": l2_t,
    }


def mass_per_timestep_np(U):
    return np.mean(U, axis=(1, 2))


# ============================================================
# 7. WINDOWS / TEMPORAL BASIS
# ============================================================

def make_window_start_indices(nt, K, stride):
    if nt <= K:
        raise ValueError(f"Need Nt > K. Got Nt={nt}, K={K}")
    return list(range(0, nt - K, stride))


def build_window_tensors(U, starts):
    n, nx, ny, nt = U.shape
    n_starts = len(starts)
    u0 = np.empty((n * n_starts, nx, ny), dtype=np.float32)
    Uf = np.empty((n * n_starts, nx, ny, WINDOW_K), dtype=np.float32)
    q = 0
    for i in range(n):
        for s in starts:
            u0[q] = U[i, :, :, s]
            Uf[q] = U[i, :, :, s + 1:s + WINDOW_K + 1]
            q += 1
    return u0, Uf


def compute_time_pod_basis_from_windows(u_start, U_future, n_modes):
    R = U_future - u_start[:, :, :, None]
    X = R.reshape(-1, WINDOW_K).astype(np.float64)

    print("\nComputing local time POD basis from TRAIN windows")
    print("------------------------------------------------")
    print("POD matrix shape:", X.shape)

    _, S, Vt = np.linalg.svd(X, full_matrices=False)

    r = min(n_modes, Vt.shape[0])
    basis = Vt[:r].T.astype(np.float32)

    energy = S ** 2
    ratio = float(np.sum(energy[:r]) / (np.sum(energy) + 1e-12))

    print(f"Effective rank  : {r}")
    print(f"Energy captured : {ratio:.10f}")
    print("Singular values :", S[:min(20, len(S))])

    return basis, ratio, S.astype(np.float32)


def make_random_orthonormal_basis(K, M):
    Q, _ = np.linalg.qr(np.random.randn(K, M))
    return Q[:, :M].astype(np.float32)


def reconstruct_window_torch(u_start, coeff, basis, enforce_mean=False):
    R = torch.einsum("bxym,km->bxyk", coeff, basis)
    Uf = u_start.unsqueeze(-1) + R
    if enforce_mean:
        mean0 = u_start.mean(dim=(1, 2), keepdim=True).unsqueeze(-1)
        Uf = Uf + mean0 - Uf.mean(dim=(1, 2), keepdim=True)
    return Uf


def enforce_mean_future(U_future, u_start):
    mean0 = u_start.mean(dim=(1, 2), keepdim=True).unsqueeze(-1)
    return U_future + mean0 - U_future.mean(dim=(1, 2), keepdim=True)


# ============================================================
# 8. DATASETS / NORMALIZER
# ============================================================

class WindowDataset2d(Dataset):
    def __init__(self, u_start, U_future):
        self.u_start = torch.tensor(u_start, dtype=torch.float32)
        self.U_future = torch.tensor(U_future, dtype=torch.float32)

    def __len__(self):
        return self.u_start.shape[0]

    def __getitem__(self, idx):
        return {"u_start": self.u_start[idx], "U_future": self.U_future[idx]}


class TrajectoryDataset2d(Dataset):
    """
    Dataset for the non-windowed FNO baseline.
    """
    def __init__(self, U):
        U = U.astype(np.float32, copy=False)
        self.u0 = torch.tensor(U[:, :, :, 0], dtype=torch.float32)
        self.U_future = torch.tensor(U[:, :, :, 1:], dtype=torch.float32)

    def __len__(self):
        return self.u0.shape[0]

    def __getitem__(self, idx):
        return {"u0": self.u0[idx], "U_future": self.U_future[idx]}


class GaussianNormalizer:
    def __init__(self, x, eps=1e-6):
        self.mean = x.mean()
        self.std = x.std()
        self.eps = eps

    def encode(self, x):
        return (x - self.mean) / (self.std + self.eps)

    def decode(self, x):
        return x * (self.std + self.eps) + self.mean

    def to(self, device_):
        self.mean = self.mean.to(device_)
        self.std = self.std.to(device_)
        return self


# ============================================================
# 9. MODELS
# ============================================================

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes_x = modes_x
        self.modes_y = modes_y

        scale = 1.0 / np.sqrt(in_channels * out_channels)

        self.weights_pos = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes_x, modes_y, dtype=torch.cfloat)
        )
        self.weights_neg = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes_x, modes_y, dtype=torch.cfloat)
        )

    @staticmethod
    def compl_mul2d(a, b):
        return torch.einsum("bixy,ioxy->boxy", a, b)

    def forward(self, x):
        B, C, nx, ny = x.shape
        x_ft = torch.fft.rfft2(x, norm="ortho")

        out_ft = torch.zeros(
            B,
            self.out_channels,
            nx,
            ny // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )

        mx = min(self.modes_x, nx)
        my = min(self.modes_y, ny // 2 + 1)

        if mx > 0 and my > 0:
            out_ft[:, :, :mx, :my] = self.compl_mul2d(
                x_ft[:, :, :mx, :my],
                self.weights_pos[:, :, :mx, :my],
            )
            out_ft[:, :, -mx:, :my] = self.compl_mul2d(
                x_ft[:, :, -mx:, :my],
                self.weights_neg[:, :, :mx, :my],
            )

        return torch.fft.irfft2(out_ft, s=(nx, ny), norm="ortho")


class LocalOperator2d(nn.Module):
    def __init__(self, width, radius):
        super().__init__()
        self.op = nn.Conv2d(
            width,
            width,
            kernel_size=2 * radius + 1,
            padding=radius,
            padding_mode="circular",
            bias=True,
        )

    def forward(self, z):
        return self.op(z)


class FourierOperator2d(nn.Module):
    def __init__(self, width, modes_x, modes_y):
        super().__init__()
        self.op = SpectralConv2d(width, width, modes_x, modes_y)

    def forward(self, z):
        return self.op(z)


class ExactLKLAlternationLayer2d(nn.Module):
    def __init__(self, width, modes_x, modes_y, local_radius):
        super().__init__()
        self.L_l1 = LocalOperator2d(width, local_radius)
        self.K_l = FourierOperator2d(width, modes_x, modes_y)
        self.L_l2 = LocalOperator2d(width, local_radius)
        self.sigma = nn.GELU()

    def forward(self, v_l):
        v_l_1 = v_l + self.sigma(self.L_l1(v_l))
        v_l_2 = v_l_1 + self.sigma(torch.tanh(self.K_l(v_l_1)))
        return v_l_2 + self.sigma(self.L_l2(v_l_2))


def make_grid_buffer():
    x = torch.tensor(x_grid_np, dtype=torch.float32)
    y = torch.tensor(y_grid_np, dtype=torch.float32)
    xn = (x - x.min()) / (x.max() - x.min() + 1e-12)
    yn = (y - y.min()) / (y.max() - y.min() + 1e-12)
    X, Y = torch.meshgrid(xn, yn, indexing="ij")
    grid = torch.stack(
        [
            X,
            Y,
            torch.sin(2 * np.pi * X),
            torch.cos(2 * np.pi * X),
            torch.sin(2 * np.pi * Y),
            torch.cos(2 * np.pi * Y),
        ],
        dim=-1,
    )
    return grid.unsqueeze(0)


class AlternatingFNOEncoder2d(nn.Module):
    def __init__(self, width, n_layers, modes_x, modes_y, local_radius):
        super().__init__()
        self.register_buffer("grid", make_grid_buffer(), persistent=False)
        self.input_proj = nn.Linear(1 + 6, width)
        self.blocks = nn.ModuleList([
            ExactLKLAlternationLayer2d(width, modes_x, modes_y, local_radius)
            for _ in range(n_layers)
        ])

    def forward(self, u_start_norm):
        B, nx, ny = u_start_norm.shape
        grid = self.grid.repeat(B, 1, 1, 1)
        z = torch.cat([u_start_norm.unsqueeze(-1), grid], dim=-1)
        z = self.input_proj(z).permute(0, 3, 1, 2).contiguous()
        for block in self.blocks:
            z = block(z)
        return z.permute(0, 2, 3, 1).contiguous()


class TBAWFNO2d(nn.Module):
    def __init__(self, n_modes):
        super().__init__()
        self.encoder = AlternatingFNOEncoder2d(WIDTH, N_LAYERS, MODES_X, MODES_Y, LOCAL_RADIUS)
        self.fc1 = nn.Linear(WIDTH, 128)
        self.fc2 = nn.Linear(128, n_modes)
        self.act = nn.GELU()

    def forward(self, u_start_norm):
        z = self.encoder(u_start_norm)
        return self.fc2(self.act(self.fc1(z))).contiguous()


class AWFNO2d(nn.Module):
    def __init__(self, window_k):
        super().__init__()
        self.encoder = AlternatingFNOEncoder2d(WIDTH, N_LAYERS, MODES_X, MODES_Y, LOCAL_RADIUS)
        self.fc1 = nn.Linear(WIDTH, 128)
        self.fc2 = nn.Linear(128, window_k)
        self.act = nn.GELU()

    def forward(self, u_start_norm):
        z = self.encoder(u_start_norm)
        return self.fc2(self.act(self.fc1(z))).contiguous()


class WFNO2d(nn.Module):
    """
    Windowed FNO baseline: standard FNO-style backbone + K-step readout.
    """
    def __init__(self, window_k):
        super().__init__()
        self.register_buffer("grid", make_grid_buffer(), persistent=False)
        self.fc0 = nn.Linear(1 + 6, WIDTH)
        self.convs = nn.ModuleList([SpectralConv2d(WIDTH, WIDTH, MODES_X, MODES_Y) for _ in range(N_LAYERS)])
        self.ws = nn.ModuleList([nn.Conv2d(WIDTH, WIDTH, 1) for _ in range(N_LAYERS)])
        self.fc1 = nn.Linear(WIDTH, 128)
        self.fc2 = nn.Linear(128, window_k)

    def forward(self, u_start_norm):
        B, nx, ny = u_start_norm.shape
        grid = self.grid.repeat(B, 1, 1, 1)
        z = torch.cat([u_start_norm.unsqueeze(-1), grid], dim=-1)
        z = self.fc0(z).permute(0, 3, 1, 2).contiguous()

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            z = conv(z) + w(z)
            if i != len(self.convs) - 1:
                z = F.gelu(z)

        z = z.permute(0, 2, 3, 1).contiguous()
        return self.fc2(F.gelu(self.fc1(z))).contiguous()


class FNOFull2d(nn.Module):
    """
    Non-windowed FNO baseline:
        u0 -> (u1,...,u_{Nt-1})

    No recursive rollout.
    """
    def __init__(self, n_future):
        super().__init__()
        self.register_buffer("grid", make_grid_buffer(), persistent=False)
        self.fc0 = nn.Linear(1 + 6, WIDTH)
        self.convs = nn.ModuleList([SpectralConv2d(WIDTH, WIDTH, MODES_X, MODES_Y) for _ in range(N_LAYERS)])
        self.ws = nn.ModuleList([nn.Conv2d(WIDTH, WIDTH, 1) for _ in range(N_LAYERS)])
        self.fc1 = nn.Linear(WIDTH, 128)
        self.fc2 = nn.Linear(128, n_future)

    def forward(self, u0_norm):
        B, nx, ny = u0_norm.shape
        grid = self.grid.repeat(B, 1, 1, 1)
        z = torch.cat([u0_norm.unsqueeze(-1), grid], dim=-1)
        z = self.fc0(z).permute(0, 3, 1, 2).contiguous()

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            z = conv(z) + w(z)
            if i != len(self.convs) - 1:
                z = F.gelu(z)

        z = z.permute(0, 2, 3, 1).contiguous()
        return self.fc2(F.gelu(self.fc1(z))).contiguous()


def make_tb_awfno_predict(model, u_normalizer, basis_torch):
    def predict(u_start):
        coeff = model(u_normalizer.encode(u_start))
        Uf = reconstruct_window_torch(u_start, coeff, basis_torch, ENFORCE_MEAN)
        return coeff, Uf
    return predict


def make_direct_predict(model, u_normalizer, future_normalizer, enforce_mean=False):
    def predict(u_start):
        Uf = future_normalizer.decode(model(u_normalizer.encode(u_start)))
        if enforce_mean:
            Uf = enforce_mean_future(Uf, u_start)
        return Uf
    return predict


def make_fno_full_predict(model, u0_normalizer, full_future_normalizer, enforce_mean=False):
    def predict(u0):
        U_future = full_future_normalizer.decode(model(u0_normalizer.encode(u0)))
        if enforce_mean:
            U_future = enforce_mean_future(U_future, u0)
        return U_future
    return predict


# ============================================================
# 10. LOSSES / TRAINING
# ============================================================

def data_loss_fn(pred, true):
    if LOSS_KIND == "mse":
        return F.mse_loss(pred, true)
    if LOSS_KIND == "huber":
        return F.smooth_l1_loss(pred, true, beta=HUBER_BETA)
    raise ValueError(LOSS_KIND)


def normalized_loss(pred, true, normalizer):
    if NORMALIZE_LOSS:
        return data_loss_fn(normalizer.encode(pred), normalizer.encode(true))
    return data_loss_fn(pred, true)


def spatial_gradient_periodic(u):
    ux = (torch.roll(u, -1, dims=1) - torch.roll(u, 1, dims=1)) / (2.0 * DX)
    uy = (torch.roll(u, -1, dims=2) - torch.roll(u, 1, dims=2)) / (2.0 * DY)
    return ux, uy


def gradient_loss(pred, true, normalizer):
    pred_n = normalizer.encode(pred) if NORMALIZE_LOSS else pred
    true_n = normalizer.encode(true) if NORMALIZE_LOSS else true
    px, py = spatial_gradient_periodic(pred_n)
    tx, ty = spatial_gradient_periodic(true_n)
    return F.smooth_l1_loss(px, tx, beta=HUBER_BETA) + F.smooth_l1_loss(py, ty, beta=HUBER_BETA)


def window_loss(Uf_pred, Uf_true, future_normalizer):
    """
    Loss for TB-AWFNO, AWFNO, and WFNO only.
    Includes final snapshot of each local window.
    """
    loss_recon = normalized_loss(Uf_pred, Uf_true, future_normalizer)
    loss_final = normalized_loss(Uf_pred[:, :, :, -1], Uf_true[:, :, :, -1], future_normalizer)
    if GRAD_LOSS_WEIGHT > 0.0:
        loss_grad = gradient_loss(Uf_pred, Uf_true, future_normalizer)
    else:
        loss_grad = 0.0
    return loss_recon + FINAL_LOSS_WEIGHT * loss_final + GRAD_LOSS_WEIGHT * loss_grad


def fno_full_loss(U_future_pred, U_future_true, full_future_normalizer):
    """
    Loss for the non-windowed FNO only.
    No extra final-time term.
    """
    return normalized_loss(U_future_pred, U_future_true, full_future_normalizer)


def train_epoch_tb_awfno(model, loader, optimizer, predict, future_normalizer):
    model.train()
    total_loss = total_l2 = n_seen = 0.0

    for batch in loader:
        u0 = batch["u_start"].to(device)
        Uf_true = batch["U_future"].to(device)

        optimizer.zero_grad(set_to_none=True)
        _, Uf_pred = predict(u0)
        loss = window_loss(Uf_pred, Uf_true, future_normalizer)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(Uf_pred.detach(), Uf_true).mean().item() * bs
        n_seen += bs

    return {"loss": total_loss / max(n_seen, 1), "abs_l2": total_l2 / max(n_seen, 1)}


@torch.no_grad()
def eval_epoch_tb_awfno(model, loader, predict, future_normalizer):
    model.eval()
    total_loss = total_l2 = total_final = n_seen = 0.0

    for batch in loader:
        u0 = batch["u_start"].to(device)
        Uf_true = batch["U_future"].to(device)
        _, Uf_pred = predict(u0)
        loss = window_loss(Uf_pred, Uf_true, future_normalizer)

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(Uf_pred, Uf_true).mean().item() * bs
        total_final += abs_l2_field_batch(Uf_pred[:, :, :, -1], Uf_true[:, :, :, -1]).mean().item() * bs
        n_seen += bs

    return {
        "loss": total_loss / max(n_seen, 1),
        "abs_l2": total_l2 / max(n_seen, 1),
        "final_l2": total_final / max(n_seen, 1),
    }


def fit_tb_awfno_model(model, train_loader, val_loader, predict, future_normalizer):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val, best_state = float("inf"), None
    hist = {"train_loss": [], "train_l2": [], "val_loss": [], "val_l2": [], "val_final_l2": []}
    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        tr = train_epoch_tb_awfno(model, train_loader, optimizer, predict, future_normalizer)
        va = eval_epoch_tb_awfno(model, val_loader, predict, future_normalizer)
        scheduler.step()

        hist["train_loss"].append(tr["loss"])
        hist["train_l2"].append(tr["abs_l2"])
        hist["val_loss"].append(va["loss"])
        hist["val_l2"].append(va["abs_l2"])
        hist["val_final_l2"].append(va["final_l2"])

        if va["abs_l2"] < best_val:
            best_val, best_state = va["abs_l2"], clean_state_dict(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            print(
                f"[TB-AWFNO] {ep:04d}/{EPOCHS} | "
                f"train loss {tr['loss']:.6e} | train L2 {tr['abs_l2']:.6e} | "
                f"val loss {va['loss']:.6e} | val L2 {va['abs_l2']:.6e} | "
                f"val final {va['final_l2']:.6e}"
            )

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    return {"model": model, "best_val_l2": best_val, "train_time_sec": time.time() - t0, "history": hist}


def train_epoch_direct(model, loader, optimizer, predict, future_normalizer, kind):
    model.train()
    total_loss = total_l2 = n_seen = 0.0

    for batch in loader:
        u0 = batch["u_start"].to(device)
        Uf_true = batch["U_future"].to(device)

        optimizer.zero_grad(set_to_none=True)
        Uf_pred = predict(u0)

        # AWFNO and WFNO both use the same local-window loss with final-window term.
        loss = window_loss(Uf_pred, Uf_true, future_normalizer)

        loss.backward()
        if kind == "awfno":
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(Uf_pred.detach(), Uf_true).mean().item() * bs
        n_seen += bs

    return {"loss": total_loss / max(n_seen, 1), "abs_l2": total_l2 / max(n_seen, 1)}


@torch.no_grad()
def eval_epoch_direct(model, loader, predict, future_normalizer, kind):
    model.eval()
    total_loss = total_l2 = total_final = n_seen = 0.0

    for batch in loader:
        u0 = batch["u_start"].to(device)
        Uf_true = batch["U_future"].to(device)
        Uf_pred = predict(u0)

        loss = window_loss(Uf_pred, Uf_true, future_normalizer)

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(Uf_pred, Uf_true).mean().item() * bs
        total_final += abs_l2_field_batch(Uf_pred[:, :, :, -1], Uf_true[:, :, :, -1]).mean().item() * bs
        n_seen += bs

    return {
        "loss": total_loss / max(n_seen, 1),
        "abs_l2": total_l2 / max(n_seen, 1),
        "final_l2": total_final / max(n_seen, 1),
    }


def fit_direct_model(model, train_loader, val_loader, predict, future_normalizer, name, kind):
    if kind == "awfno":
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(EPOCHS // 3, 1), gamma=0.5)

    best_val, best_state = float("inf"), None
    hist = {"train_loss": [], "train_l2": [], "val_loss": [], "val_l2": [], "val_final_l2": []}
    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        tr = train_epoch_direct(model, train_loader, optimizer, predict, future_normalizer, kind)
        va = eval_epoch_direct(model, val_loader, predict, future_normalizer, kind)
        scheduler.step()

        hist["train_loss"].append(tr["loss"])
        hist["train_l2"].append(tr["abs_l2"])
        hist["val_loss"].append(va["loss"])
        hist["val_l2"].append(va["abs_l2"])
        hist["val_final_l2"].append(va["final_l2"])

        if va["abs_l2"] < best_val:
            best_val, best_state = va["abs_l2"], clean_state_dict(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            print(
                f"[{name}] {ep:04d}/{EPOCHS} | "
                f"train loss {tr['loss']:.6e} | train L2 {tr['abs_l2']:.6e} | "
                f"val loss {va['loss']:.6e} | val L2 {va['abs_l2']:.6e} | "
                f"val final {va['final_l2']:.6e}"
            )

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    return {"model": model, "best_val_l2": best_val, "train_time_sec": time.time() - t0, "history": hist}


def train_epoch_fno_full(model, loader, optimizer, predict, full_future_normalizer):
    model.train()
    total_loss = total_l2 = n_seen = 0.0

    for batch in loader:
        u0 = batch["u0"].to(device)
        U_future_true = batch["U_future"].to(device)

        optimizer.zero_grad(set_to_none=True)
        U_future_pred = predict(u0)

        # Full FNO gets full-trajectory MSE only. No extra final term.
        loss = fno_full_loss(U_future_pred, U_future_true, full_future_normalizer)

        loss.backward()
        optimizer.step()

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(U_future_pred.detach(), U_future_true).mean().item() * bs
        n_seen += bs

    return {"loss": total_loss / max(n_seen, 1), "abs_l2": total_l2 / max(n_seen, 1)}


@torch.no_grad()
def eval_epoch_fno_full(model, loader, predict, full_future_normalizer):
    model.eval()
    total_loss = total_l2 = total_final = n_seen = 0.0

    for batch in loader:
        u0 = batch["u0"].to(device)
        U_future_true = batch["U_future"].to(device)
        U_future_pred = predict(u0)

        loss = fno_full_loss(U_future_pred, U_future_true, full_future_normalizer)

        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(U_future_pred, U_future_true).mean().item() * bs
        total_final += abs_l2_field_batch(U_future_pred[:, :, :, -1], U_future_true[:, :, :, -1]).mean().item() * bs
        n_seen += bs

    return {
        "loss": total_loss / max(n_seen, 1),
        "abs_l2": total_l2 / max(n_seen, 1),
        "final_l2": total_final / max(n_seen, 1),
    }


def fit_fno_full_model(model, train_loader, val_loader, predict, full_future_normalizer):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(EPOCHS // 3, 1), gamma=0.5)

    best_val = float("inf")
    best_state = None
    hist = {"train_loss": [], "train_l2": [], "val_loss": [], "val_l2": [], "val_final_l2": []}
    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        tr = train_epoch_fno_full(model, train_loader, optimizer, predict, full_future_normalizer)
        va = eval_epoch_fno_full(model, val_loader, predict, full_future_normalizer)
        scheduler.step()

        hist["train_loss"].append(tr["loss"])
        hist["train_l2"].append(tr["abs_l2"])
        hist["val_loss"].append(va["loss"])
        hist["val_l2"].append(va["abs_l2"])
        hist["val_final_l2"].append(va["final_l2"])

        if va["abs_l2"] < best_val:
            best_val = va["abs_l2"]
            best_state = clean_state_dict(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            print(
                f"[FNO full] {ep:04d}/{EPOCHS} | "
                f"train loss {tr['loss']:.6e} | train L2 {tr['abs_l2']:.6e} | "
                f"val loss {va['loss']:.6e} | val L2 {va['abs_l2']:.6e} | "
                f"val final {va['final_l2']:.6e}"
            )

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    return {"model": model, "best_val_l2": best_val, "train_time_sec": time.time() - t0, "history": hist}


# ============================================================
# 11. STRICT ROLLOUT / FULL FNO PREDICTION
# ============================================================

def check_rollout_input_gap(u_current, U_true_batch, cursor, model_name):
    if not RUN_ROLLOUT_INPUT_GAP_CHECK or cursor <= 0:
        return None

    true_current = torch.tensor(U_true_batch[:, :, :, cursor], dtype=torch.float32, device=device)
    gap = abs_l2_field_batch(u_current, true_current).mean().item()

    print(f"  CHECK {model_name}: input gap from true state at cursor {cursor}: {gap:.6e}")

    if gap < INPUT_GAP_TOL:
        raise RuntimeError(
            f"Possible cheating in {model_name}: rollout input equals true state at cursor {cursor}."
        )

    return gap


@torch.no_grad()
def honest_rollout_tb_awfno(model, U_test_np, predict, print_diagnostics=True):
    model.eval()
    n, nx, ny, nt = U_test_np.shape
    U_roll = np.zeros_like(U_test_np, dtype=np.float32)

    print("\nSTRICT rollout: TB-AWFNO")

    for i0 in range(0, n, max(1, BATCH_SIZE)):
        i1 = min(i0 + max(1, BATCH_SIZE), n)
        U_batch = U_test_np[i0:i1]

        u_current = torch.tensor(U_batch[:, :, :, 0], dtype=torch.float32, device=device)
        U_roll[i0:i1, :, :, 0] = U_batch[:, :, :, 0]

        cursor = 0
        block_id = 0

        while cursor < nt - 1:
            check_rollout_input_gap(u_current, U_batch, cursor, "TB-AWFNO")
            _, Uf_pred = predict(u_current)

            use_k = min(WINDOW_K, nt - 1 - cursor)
            U_roll[i0:i1, :, :, cursor + 1:cursor + 1 + use_k] = (
                Uf_pred[:, :, :, :use_k].detach().cpu().numpy()
            )

            u_next = Uf_pred[:, :, :, use_k - 1].detach()
            cursor_next = cursor + use_k

            if print_diagnostics:
                true_state = torch.tensor(U_batch[:, :, :, cursor_next], dtype=torch.float32, device=device)
                terminal_l2 = abs_l2_field_batch(u_next, true_state).mean().item()
                print(
                    f"TB-AWFNO batch {i0:04d}:{i1:04d} block {block_id:03d} | "
                    f"{cursor:04d}->{cursor_next:04d} | terminal L2 = {terminal_l2:.6e}"
                )

            u_current = u_next
            cursor = cursor_next
            block_id += 1

    return U_roll


@torch.no_grad()
def honest_rollout_direct(model, U_test_np, predict, model_name, print_diagnostics=True):
    model.eval()
    n, nx, ny, nt = U_test_np.shape
    U_roll = np.zeros_like(U_test_np, dtype=np.float32)

    print(f"\nSTRICT rollout: {model_name}")

    for i0 in range(0, n, max(1, BATCH_SIZE)):
        i1 = min(i0 + max(1, BATCH_SIZE), n)
        U_batch = U_test_np[i0:i1]

        u_current = torch.tensor(U_batch[:, :, :, 0], dtype=torch.float32, device=device)
        U_roll[i0:i1, :, :, 0] = U_batch[:, :, :, 0]

        cursor = 0
        block_id = 0

        while cursor < nt - 1:
            check_rollout_input_gap(u_current, U_batch, cursor, model_name)
            Uf_pred = predict(u_current)

            use_k = min(WINDOW_K, nt - 1 - cursor)
            U_roll[i0:i1, :, :, cursor + 1:cursor + 1 + use_k] = (
                Uf_pred[:, :, :, :use_k].detach().cpu().numpy()
            )

            u_next = Uf_pred[:, :, :, use_k - 1].detach()
            cursor_next = cursor + use_k

            if print_diagnostics:
                true_state = torch.tensor(U_batch[:, :, :, cursor_next], dtype=torch.float32, device=device)
                terminal_l2 = abs_l2_field_batch(u_next, true_state).mean().item()
                print(
                    f"{model_name} batch {i0:04d}:{i1:04d} block {block_id:03d} | "
                    f"{cursor:04d}->{cursor_next:04d} | terminal L2 = {terminal_l2:.6e}"
                )

            u_current = u_next
            cursor = cursor_next
            block_id += 1

    return U_roll


@torch.no_grad()
def honest_predict_fno_full(model, U_test_np, predict, print_diagnostics=True):
    """
    Non-windowed FNO evaluation:
        use true u0 only; predict full future in one call.
    """
    model.eval()
    n, nx, ny, nt = U_test_np.shape
    U_pred = np.zeros_like(U_test_np, dtype=np.float32)
    U_pred[:, :, :, 0] = U_test_np[:, :, :, 0]

    print("\nSTRICT full prediction: FNO")

    for i0 in range(0, n, max(1, BATCH_SIZE)):
        i1 = min(i0 + max(1, BATCH_SIZE), n)
        u0 = torch.tensor(U_test_np[i0:i1, :, :, 0], dtype=torch.float32, device=device)

        U_future_pred = predict(u0)
        U_pred[i0:i1, :, :, 1:] = U_future_pred.detach().cpu().numpy()

        if print_diagnostics:
            true_final = torch.tensor(U_test_np[i0:i1, :, :, -1], dtype=torch.float32, device=device)
            pred_final = U_future_pred[:, :, :, -1]
            final_l2 = abs_l2_field_batch(pred_final, true_final).mean().item()
            print(f"FNO batch {i0:04d}:{i1:04d} | u0 -> full future | final L2 = {final_l2:.6e}")

    return U_pred


def persistence_rollout(U_test_np):
    U_roll = np.zeros_like(U_test_np, dtype=np.float32)
    U_roll[:, :, :, :] = U_test_np[:, :, :, 0:1]
    return U_roll


# ============================================================
# 12. PLOTS / PRINTING
# ============================================================

def get_plot_time_indices(nt):
    out = []
    for item in PLOT_TIME_INDICES_MANUAL:
        out.append(nt - 1 if item == "final" else int(item))

    clean = []
    for k in out:
        if 0 <= k < nt and k not in clean:
            clean.append(k)

    return clean or [nt - 1]


def plot_training_curves(history, name, tag):
    if not SAVE_PLOTS or not MAKE_TRAINING_CURVE_PLOTS:
        return

    plt.figure(figsize=(8, 5))
    plt.semilogy(history["train_loss"], label="train loss")
    plt.semilogy(history["val_loss"], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{name}: training loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    p = PLOT_DIR / f"{tag}_{name}_loss_curves.png"
    plt.savefig(p, dpi=PLOT_DPI)
    if SHOW_PLOTS:
        plt.show()
    plt.close()
    print("Saved:", p)

    plt.figure(figsize=(8, 5))
    plt.semilogy(history["train_l2"], label="train L2")
    plt.semilogy(history["val_l2"], label="val L2")
    plt.semilogy(history["val_final_l2"], label="val final L2")
    plt.xlabel("epoch")
    plt.ylabel("absolute physical L2")
    plt.title(f"{name}: physical L2")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    p = PLOT_DIR / f"{tag}_{name}_l2_curves.png"
    plt.savefig(p, dpi=PLOT_DPI)
    if SHOW_PLOTS:
        plt.show()
    plt.close()
    print("Saved:", p)


def plot_l2_per_time_compare(stats_dict, tag):
    if not SAVE_PLOTS or not MAKE_L2_PER_TIME_PLOT:
        return

    # Remove persistence baseline from this plot
    stats_dict = {
        name: stats
        for name, stats in stats_dict.items()
        if name.lower() not in ["persistence", "persistance"]
    }

    plt.figure(figsize=(8, 5))

    for name, stats in stats_dict.items():
        plt.plot(
            t_grid,
            stats["mean_t"],
            linewidth=2,
            linestyle="-",
            label=f"{name} mean",
        )

        plt.plot(
            t_grid,
            stats["median_t"],
            linewidth=2,
            linestyle="--",
            label=f"{name} median",
        )

    plt.xlabel("time")
    plt.ylabel(r"$E_{L^2(\Omega)}$")
    plt.grid(True, alpha=0.3)
    plt.legend(ncol=2, fontsize=9)
    plt.tight_layout()

    p = PLOT_DIR / f"{tag}_mean_median_L2_per_timestep.png"
    plt.savefig(p, dpi=PLOT_DPI)

    if SHOW_PLOTS:
        plt.show()

    plt.close()
    print("Saved:", p)


def plot_field_panels(U_true, rollouts_dict, tag):
    if not SAVE_PLOTS or not MAKE_FIELD_PLOTS:
        print("Field plots skipped. Set MAKE_FIELD_PLOTS=1 to enable.")
        return

    out_dir = PLOT_DIR / f"{tag}_field_panels"
    out_dir.mkdir(parents=True, exist_ok=True)

    time_ids = get_plot_time_indices(U_true.shape[-1])
    n_plot = min(MAX_TEST_CASES_TO_PLOT, U_true.shape[0])

    # Remove Persistence / Persistance from field plots
    rollouts_dict = {
        name: arr
        for name, arr in rollouts_dict.items()
        if name.lower() not in ["persistence", "persistance"]
    }

    model_names = list(rollouts_dict.keys())

    print("\nPlotting field panels")
    print("---------------------")
    print("output dir:", out_dir)
    print("test cases:", n_plot)
    print("times     :", time_ids)
    print("models    :", model_names)

    for i in range(n_plot):
        for tidx in time_ids:
            true_field = U_true[i, :, :, tidx]

            state_fields = [true_field]
            for name in model_names:
                pred_field = rollouts_dict[name][i, :, :, tidx]
                state_fields.append(pred_field)

            vmin = min(float(f.min()) for f in state_fields)
            vmax = max(float(f.max()) for f in state_fields)

            # Only true + prediction panels
            ncols = 1 + len(model_names)

            fig, axes = plt.subplots(
                1,
                ncols,
                figsize=(3.0 * ncols, 3.2),
                constrained_layout=True,
            )

            axes = np.asarray(axes).ravel()

            col = 0

            im_state = axes[col].imshow(
                true_field.T,
                origin="lower",
                vmin=vmin,
                vmax=vmax,
                aspect="equal",
            )
            axes[col].set_title("true")
            axes[col].set_xticks([])
            axes[col].set_yticks([])
            col += 1

            for name in model_names:
                pred_field = rollouts_dict[name][i, :, :, tidx]

                axes[col].imshow(
                    pred_field.T,
                    origin="lower",
                    vmin=vmin,
                    vmax=vmax,
                    aspect="equal",
                )
                axes[col].set_title(f"{name} pred")
                axes[col].set_xticks([])
                axes[col].set_yticks([])
                col += 1

            fig.colorbar(
                im_state,
                ax=axes.tolist(),
                shrink=0.7,
            )

            fig.suptitle(
                f"test {i}, t index {tidx}, t={float(t_grid[tidx]):.4f}"
            )

            p = out_dir / f"test_{i:03d}_t{tidx:04d}.png"
            plt.savefig(p, dpi=PLOT_DPI)

            if SHOW_PLOTS:
                plt.show()

            plt.close(fig)
            print("Saved:", p)


def print_l2_results(tb_awfno_stats, awfno_stats, wfno_stats, fno_stats, persistence_stats, train_times=None):
    print("\n================ ABSOLUTE PHYSICAL L2 TEST RESULTS ================")
    train_times = train_times or {"tb_awfno": None, "awfno": None, "wfno": None, "fno": None}

    def mt(k):
        return "" if train_times.get(k) is None else f" | train time {train_times[k]:.2f}s"

    rows = [
        ("TB-AWFNO", tb_awfno_stats, "tb_awfno"),
        ("AWFNO", awfno_stats, "awfno"),
        ("WFNO", wfno_stats, "wfno"),
        ("FNO", fno_stats, "fno"),
        ("Persistence", persistence_stats, "persistence"),
    ]

    for label, stats, key in rows:
        print(
            f"{label:<11} | "
            f"L2(full) mean {stats['full_l2_mean']:.6e} | "
            f"std {stats['full_l2_std']:.6e} | "
            f"median {stats['full_l2_median']:.6e} | "
            f"L2(final) mean {stats['final_l2_mean']:.6e} | "
            f"final median {stats['final_l2_median']:.6e}"
            f"{mt(key)}"
        )

    scores = {
        "TB-AWFNO": tb_awfno_stats["full_l2_mean"],
        "AWFNO": awfno_stats["full_l2_mean"],
        "WFNO": wfno_stats["full_l2_mean"],
        "FNO": fno_stats["full_l2_mean"],
        "Persistence": persistence_stats["full_l2_mean"],
    }
    winner = min(scores, key=scores.get)
    print(f"\nWinner by mean full-trajectory absolute L2: {winner}")
    print("===============================================================\n")
    return winner


# ============================================================
# 13. SAMPLE-SIZE HELPERS
# ============================================================

def make_base_tag_for_sample_size(n_train):
    data_signature = {
        "DATA_FILE_name": DATA_FILE.name,
        "DATA_FILE_size": DATA_FILE.stat().st_size if DATA_FILE.exists() else None,
    }
    return short_hash({
        "seed": SEED,
        "data_signature": data_signature,
        "SAMPLE_SIZES": SAMPLE_SIZES,
        "N_train": n_train,
        "N_VAL": N_VAL,
        "N_TEST": N_TEST,
        "Nx": Nx,
        "Ny": Ny,
        "Nt": Nt,
        "DT_PHYSICAL": DT_PHYSICAL,
        "WINDOW_K": WINDOW_K,
        "WINDOW_STRIDE": WINDOW_STRIDE,
        "N_TIME_MODES": N_TIME_MODES,
        "WIDTH": WIDTH,
        "N_LAYERS": N_LAYERS,
        "MODES_X": MODES_X,
        "MODES_Y": MODES_Y,
        "LOCAL_RADIUS": LOCAL_RADIUS,
        "LOSS_KIND": LOSS_KIND,
        "NORMALIZE_LOSS": NORMALIZE_LOSS,
        "FINAL_LOSS_WEIGHT": FINAL_LOSS_WEIGHT,
        "GRAD_LOSS_WEIGHT": GRAD_LOSS_WEIGHT,
        "ENFORCE_MEAN": ENFORCE_MEAN,
        "window_final_loss_for": ["TB-AWFNO", "AWFNO", "WFNO"],
        "fno_full_loss_has_final_extra_term": False,
        "exact_LKL": True,
        "strict_rollout": True,
    })


def build_train_val_objects_for_sample_size(n_train, U_train_pool, U_val):
    U_train = U_train_pool[:n_train]

    train_starts = make_window_start_indices(Nt, WINDOW_K, WINDOW_STRIDE)
    val_starts = make_window_start_indices(Nt, WINDOW_K, WINDOW_K)

    print("\nWindow starts per trajectory")
    print("----------------------------")
    print("train:", len(train_starts), train_starts)
    print("val  :", len(val_starts), val_starts)
    print("test : 0  <-- STRICT: no test windows created")

    u_train_win, Uf_train_win = build_window_tensors(U_train, train_starts)
    u_val_win, Uf_val_win = build_window_tensors(U_val, val_starts)

    print("\nWindow tensor shapes")
    print("--------------------")
    print("u_train_win :", u_train_win.shape)
    print("Uf_train_win:", Uf_train_win.shape)
    print("u_val_win   :", u_val_win.shape)
    print("Uf_val_win  :", Uf_val_win.shape)

    if DEBUG_SHUFFLE_TRAIN_OUTPUTS:
        print("\nWARNING: DEBUG_SHUFFLE_TRAIN_OUTPUTS=True")
        Uf_train_win = Uf_train_win[np.random.permutation(Uf_train_win.shape[0])]

    basis_np, pod_energy_ratio, singular_values = compute_time_pod_basis_from_windows(
        u_train_win,
        Uf_train_win,
        N_TIME_MODES,
    )

    if DEBUG_RANDOM_BASIS:
        print("\nWARNING: DEBUG_RANDOM_BASIS=True")
        basis_np = make_random_orthonormal_basis(WINDOW_K, N_TIME_MODES)
        pod_energy_ratio = float("nan")
        singular_values = np.ones(N_TIME_MODES, dtype=np.float32)

    basis_torch = torch.tensor(basis_np, dtype=torch.float32, device=device)

    print("\nLocal TB-AWFNO oracle L2")
    print("------------------------")
    print("Skipped. No true POD coefficients are projected or supervised.")

    train_ds = WindowDataset2d(u_train_win, Uf_train_win)
    val_ds = WindowDataset2d(u_val_win, Uf_val_win)

    cpu_gen_window = torch.Generator()
    cpu_gen_window.manual_seed(SEED + n_train)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=cpu_gen_window,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    batch0 = next(iter(train_loader))
    print("\nWindow batch shapes")
    print("-------------------")
    print("u_start :", tuple(batch0["u_start"].shape))
    print("U_future:", tuple(batch0["U_future"].shape))

    u_normalizer = GaussianNormalizer(torch.tensor(u_train_win, dtype=torch.float32)).to(device)
    future_normalizer = GaussianNormalizer(torch.tensor(Uf_train_win, dtype=torch.float32)).to(device)

    # FNO full baseline uses initial-condition normalizer, not window-input normalizer.
    fno_train_ds = TrajectoryDataset2d(U_train)
    fno_val_ds = TrajectoryDataset2d(U_val)

    cpu_gen_fno = torch.Generator()
    cpu_gen_fno.manual_seed(SEED + 100000 + n_train)

    fno_train_loader = DataLoader(
        fno_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=cpu_gen_fno,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    fno_val_loader = DataLoader(
        fno_val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    fno_batch0 = next(iter(fno_train_loader))
    print("\nFNO full batch shapes")
    print("---------------------")
    print("u0      :", tuple(fno_batch0["u0"].shape))
    print("U_future:", tuple(fno_batch0["U_future"].shape))

    full_u0_normalizer = GaussianNormalizer(torch.tensor(U_train[:, :, :, 0], dtype=torch.float32)).to(device)
    full_future_normalizer = GaussianNormalizer(torch.tensor(U_train[:, :, :, 1:], dtype=torch.float32)).to(device)

    print("\nNormalizers")
    print("-----------")
    print("window input mean/std :", float(u_normalizer.mean.cpu()), float(u_normalizer.std.cpu()))
    print("window future mean/std:", float(future_normalizer.mean.cpu()), float(future_normalizer.std.cpu()))
    print("FNO u0 mean/std       :", float(full_u0_normalizer.mean.cpu()), float(full_u0_normalizer.std.cpu()))
    print("FNO future mean/std   :", float(full_future_normalizer.mean.cpu()), float(full_future_normalizer.std.cpu()))

    return {
        "U_train": U_train,
        "u_train_win": u_train_win,
        "Uf_train_win": Uf_train_win,
        "u_val_win": u_val_win,
        "Uf_val_win": Uf_val_win,
        "basis_np": basis_np,
        "basis_torch": basis_torch,
        "pod_energy_ratio": pod_energy_ratio,
        "singular_values": singular_values,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "fno_train_loader": fno_train_loader,
        "fno_val_loader": fno_val_loader,
        "u_normalizer": u_normalizer,
        "future_normalizer": future_normalizer,
        "full_u0_normalizer": full_u0_normalizer,
        "full_future_normalizer": full_future_normalizer,
    }


def train_models_for_sample_size(n_train, U_train_pool, U_val):
    global BASE_TAG
    BASE_TAG = make_base_tag_for_sample_size(n_train)

    print("\n" + "=" * 80)
    print(f"TRAINING SAMPLE SIZE: N_train = {n_train}")
    print("=" * 80)
    print("BASE_TAG:", BASE_TAG)

    objects = build_train_val_objects_for_sample_size(n_train, U_train_pool, U_val)

    basis_torch = objects["basis_torch"]
    train_loader = objects["train_loader"]
    val_loader = objects["val_loader"]
    fno_train_loader = objects["fno_train_loader"]
    fno_val_loader = objects["fno_val_loader"]
    u_normalizer = objects["u_normalizer"]
    future_normalizer = objects["future_normalizer"]
    full_u0_normalizer = objects["full_u0_normalizer"]
    full_future_normalizer = objects["full_future_normalizer"]

    tb_awfno_model = TBAWFNO2d(N_TIME_MODES).to(device)
    awfno_model = AWFNO2d(WINDOW_K).to(device)
    wfno_model = WFNO2d(WINDOW_K).to(device)
    fno_model = FNOFull2d(Nt - 1).to(device)

    tb_awfno_predict = make_tb_awfno_predict(tb_awfno_model, u_normalizer, basis_torch)
    awfno_predict = make_direct_predict(awfno_model, u_normalizer, future_normalizer, ENFORCE_MEAN)
    wfno_predict = make_direct_predict(wfno_model, u_normalizer, future_normalizer, False)
    fno_predict = make_fno_full_predict(fno_model, full_u0_normalizer, full_future_normalizer, False)

    print("\nModels")
    print("------")
    print("TB-AWFNO: exact L-K-L encoder + temporal-basis readout")
    print("AWFNO   : exact L-K-L encoder + direct window readout")
    print("WFNO    : standard FNO encoder + direct window readout + recursive rollout")
    print("FNO     : standard FNO encoder + direct full-trajectory readout, no rollout")
    print("Window loss final term: TB-AWFNO yes, AWFNO yes, WFNO yes, FNO no")
    print("Exact L-K-L block used only by TB-AWFNO and AWFNO:")
    print("  v_l_1      = v_l   + sigma(L_l1(v_l))")
    print("  v_l_2      = v_l_1 + sigma(K_l(v_l_1))")
    print("  v_l_plus_1 = v_l_2 + sigma(L_l2(v_l_2))")
    print(f"TB-AWFNO params: {count_params(tb_awfno_model):,}")
    print(f"AWFNO params   : {count_params(awfno_model):,}")
    print(f"WFNO params    : {count_params(wfno_model):,}")
    print(f"FNO params     : {count_params(fno_model):,}")

    tb_awfno_path = MODEL_DIR / f"N{n_train}_STRICT_TB_AWFNO_CH2D_LKL_{BASE_TAG}.pt"
    awfno_path = MODEL_DIR / f"N{n_train}_STRICT_AWFNO_CH2D_LKL_{BASE_TAG}.pt"
    wfno_path = MODEL_DIR / f"N{n_train}_STRICT_WFNO_CH2D_windowed_{BASE_TAG}.pt"
    fno_path = MODEL_DIR / f"N{n_train}_STRICT_FNO_CH2D_full_{BASE_TAG}.pt"

    if tb_awfno_path.exists() and not FORCE_RETRAIN:
        print("\nLoading cached TB-AWFNO:", tb_awfno_path)
        meta = load_model_checkpoint(tb_awfno_path, tb_awfno_model)
        tb_awfno_fit = {
            "model": tb_awfno_model,
            "best_val_l2": meta.get("best_val_l2", float("nan")),
            "train_time_sec": meta.get("train_time_sec", 0.0),
            "history": meta.get("history", None),
        }
    else:
        print("\n================ Training TB-AWFNO ================")
        tb_awfno_fit = fit_tb_awfno_model(tb_awfno_model, train_loader, val_loader, tb_awfno_predict, future_normalizer)
        save_model_checkpoint(
            tb_awfno_path,
            tb_awfno_fit["model"],
            {
                "best_val_l2": tb_awfno_fit["best_val_l2"],
                "train_time_sec": tb_awfno_fit["train_time_sec"],
                "history": tb_awfno_fit["history"],
            },
        )
    if tb_awfno_fit["history"] is not None:
        plot_training_curves(tb_awfno_fit["history"], f"N{n_train}_TB_AWFNO_CH2D_LKL", BASE_TAG)

    if awfno_path.exists() and not FORCE_RETRAIN:
        print("\nLoading cached AWFNO:", awfno_path)
        meta = load_model_checkpoint(awfno_path, awfno_model)
        awfno_fit = {
            "model": awfno_model,
            "best_val_l2": meta.get("best_val_l2", float("nan")),
            "train_time_sec": meta.get("train_time_sec", 0.0),
            "history": meta.get("history", None),
        }
    else:
        print("\n================ Training AWFNO ================")
        awfno_fit = fit_direct_model(awfno_model, train_loader, val_loader, awfno_predict, future_normalizer, name="AWFNO", kind="awfno")
        save_model_checkpoint(
            awfno_path,
            awfno_fit["model"],
            {
                "best_val_l2": awfno_fit["best_val_l2"],
                "train_time_sec": awfno_fit["train_time_sec"],
                "history": awfno_fit["history"],
            },
        )
    if awfno_fit["history"] is not None:
        plot_training_curves(awfno_fit["history"], f"N{n_train}_AWFNO_CH2D_LKL", BASE_TAG)

    if wfno_path.exists() and not FORCE_RETRAIN:
        print("\nLoading cached WFNO:", wfno_path)
        meta = load_model_checkpoint(wfno_path, wfno_model)
        wfno_fit = {
            "model": wfno_model,
            "best_val_l2": meta.get("best_val_l2", float("nan")),
            "train_time_sec": meta.get("train_time_sec", 0.0),
            "history": meta.get("history", None),
        }
    else:
        print("\n================ Training WFNO ================")
        wfno_fit = fit_direct_model(wfno_model, train_loader, val_loader, wfno_predict, future_normalizer, name="WFNO", kind="wfno")
        save_model_checkpoint(
            wfno_path,
            wfno_fit["model"],
            {
                "best_val_l2": wfno_fit["best_val_l2"],
                "train_time_sec": wfno_fit["train_time_sec"],
                "history": wfno_fit["history"],
            },
        )
    if wfno_fit["history"] is not None:
        plot_training_curves(wfno_fit["history"], f"N{n_train}_WFNO_CH2D_windowed", BASE_TAG)

    if fno_path.exists() and not FORCE_RETRAIN:
        print("\nLoading cached FNO:", fno_path)
        meta = load_model_checkpoint(fno_path, fno_model)
        fno_fit = {
            "model": fno_model,
            "best_val_l2": meta.get("best_val_l2", float("nan")),
            "train_time_sec": meta.get("train_time_sec", 0.0),
            "history": meta.get("history", None),
        }
    else:
        print("\n================ Training FNO ================")
        fno_fit = fit_fno_full_model(fno_model, fno_train_loader, fno_val_loader, fno_predict, full_future_normalizer)
        save_model_checkpoint(
            fno_path,
            fno_fit["model"],
            {
                "best_val_l2": fno_fit["best_val_l2"],
                "train_time_sec": fno_fit["train_time_sec"],
                "history": fno_fit["history"],
            },
        )
    if fno_fit["history"] is not None:
        plot_training_curves(fno_fit["history"], f"N{n_train}_FNO_CH2D_full", BASE_TAG)

    return {
        "n_train": n_train,
        "base_tag": BASE_TAG,
        "objects": objects,
        "tb_awfno_model": tb_awfno_model,
        "awfno_model": awfno_model,
        "wfno_model": wfno_model,
        "fno_model": fno_model,
        "tb_awfno_predict": tb_awfno_predict,
        "awfno_predict": awfno_predict,
        "wfno_predict": wfno_predict,
        "fno_predict": fno_predict,
        "tb_awfno_fit": tb_awfno_fit,
        "awfno_fit": awfno_fit,
        "wfno_fit": wfno_fit,
        "fno_fit": fno_fit,
        "paths": {
            "tb_awfno": str(tb_awfno_path),
            "awfno": str(awfno_path),
            "wfno": str(wfno_path),
            "fno": str(fno_path),
        },
    }


def evaluate_bundle_on_same_test_set(bundle, U_test, test_idx):
    global BASE_TAG
    n_train = bundle["n_train"]
    BASE_TAG = bundle["base_tag"]

    print("\n" + "=" * 80)
    print(f"STRICT TEST EVALUATION ON COMMON TEST SET | N_train = {n_train}")
    print("=" * 80)
    print("Common test shape:", U_test.shape)
    print("Protocol: no test windows, no test POD coefficients, true u0 only.")

    print("\n================ STRICT HONEST TEST ROLLOUT: TB-AWFNO ================")
    U_roll_tb_awfno = honest_rollout_tb_awfno(
        bundle["tb_awfno_fit"]["model"],
        U_test,
        bundle["tb_awfno_predict"],
        True,
    )

    print("\n================ STRICT HONEST TEST ROLLOUT: AWFNO ================")
    U_roll_awfno = honest_rollout_direct(
        bundle["awfno_fit"]["model"],
        U_test,
        bundle["awfno_predict"],
        "AWFNO",
        True,
    )

    print("\n================ STRICT HONEST TEST ROLLOUT: WFNO ================")
    U_roll_wfno = honest_rollout_direct(
        bundle["wfno_fit"]["model"],
        U_test,
        bundle["wfno_predict"],
        "WFNO",
        True,
    )

    print("\n================ STRICT HONEST TEST EVALUATION: FNO FULL ================")
    U_roll_fno = honest_predict_fno_full(
        bundle["fno_fit"]["model"],
        U_test,
        bundle["fno_predict"],
        True,
    )

    U_roll_persistence = persistence_rollout(U_test)

    tb_awfno_stats = rollout_l2_metrics(U_roll_tb_awfno, U_test)
    awfno_stats = rollout_l2_metrics(U_roll_awfno, U_test)
    wfno_stats = rollout_l2_metrics(U_roll_wfno, U_test)
    fno_stats = rollout_l2_metrics(U_roll_fno, U_test)
    persistence_stats = rollout_l2_metrics(U_roll_persistence, U_test)

    train_times = {
        "tb_awfno": bundle["tb_awfno_fit"]["train_time_sec"],
        "awfno": bundle["awfno_fit"]["train_time_sec"],
        "wfno": bundle["wfno_fit"]["train_time_sec"],
        "fno": bundle["fno_fit"]["train_time_sec"],
    }

    winner = print_l2_results(
        tb_awfno_stats,
        awfno_stats,
        wfno_stats,
        fno_stats,
        persistence_stats,
        train_times,
    )

    rollouts_dict = {
        "TB-AWFNO": U_roll_tb_awfno,
        "AWFNO": U_roll_awfno,
        "WFNO": U_roll_wfno,
        "FNO": U_roll_fno,
        "Persistence": U_roll_persistence,
    }
    stats_dict = {
        "TB-AWFNO": tb_awfno_stats,
        "AWFNO": awfno_stats,
        "WFNO": wfno_stats,
        "FNO": fno_stats,
        "Persistence": persistence_stats,
    }

    plot_l2_per_time_compare(stats_dict, BASE_TAG)
    plot_field_panels(U_test, rollouts_dict, BASE_TAG)

    mass_true = mass_per_timestep_np(U_test)
    mass_tb_awfno = mass_per_timestep_np(U_roll_tb_awfno)
    mass_awfno = mass_per_timestep_np(U_roll_awfno)
    mass_wfno = mass_per_timestep_np(U_roll_wfno)
    mass_fno = mass_per_timestep_np(U_roll_fno)

    mass_err = {
        "tb_awfno_mass_abs_mean": float(np.mean(np.abs(mass_tb_awfno - mass_true))),
        "awfno_mass_abs_mean": float(np.mean(np.abs(mass_awfno - mass_true))),
        "wfno_mass_abs_mean": float(np.mean(np.abs(mass_wfno - mass_true))),
        "fno_mass_abs_mean": float(np.mean(np.abs(mass_fno - mass_true))),
    }

    rollout_path = None
    if SAVE_ROLLOUT_ARRAYS:
        rollout_path = RESULTS_DIR / f"N{n_train}_STRICT_rollouts_CH2D_{BASE_TAG}.npz"
        np.savez_compressed(
            rollout_path,
            U_roll_tb_awfno=U_roll_tb_awfno,
            U_roll_awfno=U_roll_awfno,
            U_roll_wfno=U_roll_wfno,
            U_roll_fno=U_roll_fno,
            U_roll_persistence=U_roll_persistence,
            U_true=U_test,
            x_grid=x_grid_np,
            y_grid=y_grid_np,
            t=t_grid,
            test_idx=test_idx,
            tb_awfno_l2_t=tb_awfno_stats["all_l2_t"],
            awfno_l2_t=awfno_stats["all_l2_t"],
            wfno_l2_t=wfno_stats["all_l2_t"],
            fno_l2_t=fno_stats["all_l2_t"],
            persistence_l2_t=persistence_stats["all_l2_t"],
        )
        rollout_path = str(rollout_path)
        print("Saved rollout arrays:", rollout_path)

    row = {
        "N_train": n_train,
        "winner": winner,

        "tb_awfno_full_l2_mean": tb_awfno_stats["full_l2_mean"],
        "awfno_full_l2_mean": awfno_stats["full_l2_mean"],
        "wfno_full_l2_mean": wfno_stats["full_l2_mean"],
        "fno_full_l2_mean": fno_stats["full_l2_mean"],
        "persistence_full_l2_mean": persistence_stats["full_l2_mean"],

        "tb_awfno_final_l2_mean": tb_awfno_stats["final_l2_mean"],
        "awfno_final_l2_mean": awfno_stats["final_l2_mean"],
        "wfno_final_l2_mean": wfno_stats["final_l2_mean"],
        "fno_final_l2_mean": fno_stats["final_l2_mean"],
        "persistence_final_l2_mean": persistence_stats["final_l2_mean"],

        "tb_awfno_best_val_l2": float(bundle["tb_awfno_fit"]["best_val_l2"]),
        "awfno_best_val_l2": float(bundle["awfno_fit"]["best_val_l2"]),
        "wfno_best_val_l2": float(bundle["wfno_fit"]["best_val_l2"]),
        "fno_best_val_l2": float(bundle["fno_fit"]["best_val_l2"]),

        "tb_awfno_train_time_sec": float(bundle["tb_awfno_fit"]["train_time_sec"]),
        "awfno_train_time_sec": float(bundle["awfno_fit"]["train_time_sec"]),
        "wfno_train_time_sec": float(bundle["wfno_fit"]["train_time_sec"]),
        "fno_train_time_sec": float(bundle["fno_fit"]["train_time_sec"]),

        "pod_energy_ratio_train_only": float(bundle["objects"]["pod_energy_ratio"]),
        **mass_err,

        "base_tag": BASE_TAG,
        "rollout_path": rollout_path,
    }

    summary_path = RESULTS_DIR / f"N{n_train}_STRICT_summary_CH2D_{BASE_TAG}.json"
    summary = {
        "task": "Cahn-Hilliard 2D strict finite-window and full-FNO benchmark",
        "data_file": str(DATA_FILE),
        "layout": "U=(N,Nx,Ny,Nt)",
        "N_train": n_train,
        "N_val": N_VAL,
        "N_test": N_TEST,
        "sample_sizes": SAMPLE_SIZES,
        "grid": {
            "Nx": Nx,
            "Ny": Ny,
            "Nt": Nt,
            "DX": DX,
            "DY": DY,
            "DT_SNAPSHOT": DT_SNAPSHOT,
        },
        "loss_protocol": {
            "TB_AWFNO": "local window MSE + final local-window snapshot MSE",
            "AWFNO": "local window MSE + final local-window snapshot MSE",
            "WFNO": "local window MSE + final local-window snapshot MSE",
            "FNO": "full future trajectory MSE only, no extra final-time term",
        },
        "strict_protocol": {
            "test_windows_created": False,
            "train_val_coefficients_projected": False,
            "test_coefficients_projected": False,
            "test_uses_only_true_initial_condition": True,
            "later_test_inputs_are_predicted": True,
            "fno_is_non_windowed": True,
            "euler_reconstruction": False,
            "relative_error_reported": False,
        },
        "row": row,
        "model_paths": bundle["paths"],
    }
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved summary:", summary_path)

    # Keep results row, release heavy arrays before next sample size.
    del U_roll_tb_awfno, U_roll_awfno, U_roll_wfno, U_roll_fno, U_roll_persistence
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


def save_learning_curve_results(rows):
    rows_sorted = sorted(rows, key=lambda r: r["N_train"])

    csv_path = RESULTS_DIR / "sample_size_sweep_CH2D_absolute_L2.csv"
    keys = list(rows_sorted[0].keys())

    with open(csv_path, "w") as f:
        f.write(",".join(keys) + "\n")
        for r in rows_sorted:
            f.write(",".join(str(r[k]) for k in keys) + "\n")
    print("Saved CSV results:", csv_path)

    json_path = RESULTS_DIR / "sample_size_sweep_CH2D_absolute_L2.json"
    with open(json_path, "w") as f:
        json.dump(rows_sorted, f, indent=2)
    print("Saved JSON results:", json_path)

    if SAVE_PLOTS:
        Ns = np.array([r["N_train"] for r in rows_sorted], dtype=np.float64)

        plt.figure(figsize=(8, 5))
        plt.plot(Ns, [r["tb_awfno_full_l2_mean"] for r in rows_sorted], marker="o", label="TB-AWFNO")
        plt.plot(Ns, [r["awfno_full_l2_mean"] for r in rows_sorted], marker="o", label="AWFNO")
        plt.plot(Ns, [r["wfno_full_l2_mean"] for r in rows_sorted], marker="o", label="WFNO")
        plt.plot(Ns, [r["fno_full_l2_mean"] for r in rows_sorted], marker="o", label="FNO")
        # plt.plot(Ns, [r["persistence_full_l2_mean"] for r in rows_sorted], marker="o", label="Persistence")
        plt.xlabel("number of training trajectories")
        plt.ylabel("mean absolute full-trajectory L2")
        plt.title("Cahn-Hilliard sample-size sweep")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        p = PLOT_DIR / "sample_size_sweep_abs_full_L2.png"
        plt.savefig(p, dpi=PLOT_DPI)
        if SHOW_PLOTS:
            plt.show()
        plt.close()
        print("Saved:", p)

        plt.figure(figsize=(8, 5))
        plt.plot(Ns, [r["tb_awfno_final_l2_mean"] for r in rows_sorted], marker="o", label="TB-AWFNO")
        plt.plot(Ns, [r["awfno_final_l2_mean"] for r in rows_sorted], marker="o", label="AWFNO")
        plt.plot(Ns, [r["wfno_final_l2_mean"] for r in rows_sorted], marker="o", label="WFNO")
        plt.plot(Ns, [r["fno_final_l2_mean"] for r in rows_sorted], marker="o", label="FNO")
        # plt.plot(Ns, [r["persistence_final_l2_mean"] for r in rows_sorted], marker="o", label="Persistence")
        plt.xlabel("number of training trajectories")
        plt.ylabel("mean absolute final-time L2")
        plt.title("Cahn-Hilliard final-time sweep")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        p = PLOT_DIR / "sample_size_sweep_abs_final_L2.png"
        plt.savefig(p, dpi=PLOT_DPI)
        if SHOW_PLOTS:
            plt.show()
        plt.close()
        print("Saved:", p)

    print("\n================ SAMPLE-SIZE SWEEP SUMMARY: ABSOLUTE FULL L2 ================")
    for r in rows_sorted:
        print(
            f"N={r['N_train']:4d} | "
            f"Abs full L2: TB-AWFNO {r['tb_awfno_full_l2_mean']:.6e} | "
            f"AWFNO {r['awfno_full_l2_mean']:.6e} | "
            f"WFNO {r['wfno_full_l2_mean']:.6e} | "
            f"FNO {r['fno_full_l2_mean']:.6e} | "
            f"Persistence {r['persistence_full_l2_mean']:.6e} | "
            f"winner {r['winner']}"
        )
    print("============================================================================\n")

    print("\n================ SAMPLE-SIZE SWEEP SUMMARY: ABSOLUTE FINAL L2 ================")
    for r in rows_sorted:
        final_scores = {
            "TB-AWFNO": r["tb_awfno_final_l2_mean"],
            "AWFNO": r["awfno_final_l2_mean"],
            "WFNO": r["wfno_final_l2_mean"],
            "FNO": r["fno_final_l2_mean"],
            "Persistence": r["persistence_final_l2_mean"],
        }
        final_winner = min(final_scores, key=final_scores.get)
        print(
            f"N={r['N_train']:4d} | "
            f"Abs final L2: TB-AWFNO {r['tb_awfno_final_l2_mean']:.6e} | "
            f"AWFNO {r['awfno_final_l2_mean']:.6e} | "
            f"WFNO {r['wfno_final_l2_mean']:.6e} | "
            f"FNO {r['fno_final_l2_mean']:.6e} | "
            f"Persistence {r['persistence_final_l2_mean']:.6e} | "
            f"winner {final_winner}"
        )
    print("=============================================================================\n")


# ============================================================
# 14. MAIN
# ============================================================

def main():
    print("\nAUDIT FLAGS")
    print("-----------")
    print(f"DATA_FILE                         : {DATA_FILE}")
    print(f"OUTPUT_ROOT                       : {OUTPUT_ROOT}")
    print(f"CACHE_ROOT                        : {CACHE_ROOT}")
    print(f"FORCE_RETRAIN                     : {FORCE_RETRAIN}")
    print(f"RUN_EXACT_HASH_OVERLAP_CHECK      : {RUN_EXACT_HASH_OVERLAP_CHECK}")
    print(f"RUN_DUPLICATE_REPORT_INSIDE_SPLITS: {RUN_DUPLICATE_REPORT_INSIDE_SPLITS}")
    print(f"RUN_ROLLOUT_INPUT_GAP_CHECK       : {RUN_ROLLOUT_INPUT_GAP_CHECK}")
    print(f"DEBUG_SHUFFLE_TRAIN_OUTPUTS       : {DEBUG_SHUFFLE_TRAIN_OUTPUTS}")
    print(f"DEBUG_RANDOM_BASIS                : {DEBUG_RANDOM_BASIS}")
    print(f"MAKE_FIELD_PLOTS                  : {MAKE_FIELD_PLOTS}")
    print(f"SAVE_ROLLOUT_ARRAYS               : {SAVE_ROLLOUT_ARRAYS}")

    print("\nExperiment protocol")
    print("-------------------")
    print("Dataset:", DATA_FILE)
    print("Internal layout: U=(N,Nx,Ny,Nt)")
    print("Training sizes:", SAMPLE_SIZES)
    print("No test windows; no test POD coefficients; true u0 only for windowed rollout.")
    print("FNO is non-windowed: u0 -> full future directly.")
    print("Windowed loss: local window + final snapshot of each window.")
    print("FNO loss: full future trajectory only, no extra final term.")
    print("Metric: absolute physical L2 only, trapezoidal in time.")
    print("Mean enforcement for TB-AWFNO/AWFNO:", ENFORCE_MEAN)

    U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid_loaded, y_grid_loaded = load_or_create_split()

    set_geometry_from_arrays(U_train_pool.shape, x_grid_loaded, y_grid_loaded, DT_PHYSICAL)

    print("\nLoaded split arrays")
    print("-------------------")
    print_array_info("U_train_pool", U_train_pool)
    print_array_info("U_val", U_val)
    print_array_info("U_test", U_test)

    print("\nSanity check train sample 0")
    print("---------------------------")
    mid = min(5, U_train_pool.shape[-1] - 1)
    print("t=0     min/max:", float(U_train_pool[0, :, :, 0].min()), float(U_train_pool[0, :, :, 0].max()))
    print(f"t={mid} min/max:", float(U_train_pool[0, :, :, mid].min()), float(U_train_pool[0, :, :, mid].max()))
    print("final   min/max:", float(U_train_pool[0, :, :, -1].min()), float(U_train_pool[0, :, :, -1].max()))
    print("mean t=0/final:", float(U_train_pool[0, :, :, 0].mean()), float(U_train_pool[0, :, :, -1].mean()))

    if not np.all(np.isfinite(U_train_pool)) or not np.all(np.isfinite(U_val)) or not np.all(np.isfinite(U_test)):
        raise ValueError("One split contains NaN/Inf.")

    if RUN_EXACT_HASH_OVERLAP_CHECK:
        check_exact_overlap(U_train_pool, U_val, "train_pool", "validation")
        check_exact_overlap(U_train_pool, U_test, "train_pool", "common_test")
        check_exact_overlap(U_val, U_test, "validation", "common_test")

    if RUN_DUPLICATE_REPORT_INSIDE_SPLITS:
        report_duplicate_trajectories(U_train_pool, "train_pool")
        report_duplicate_trajectories(U_val, "validation")
        report_duplicate_trajectories(U_test, "common_test")

    trained_bundles = []

    # Train/load all models using train/val only.
    for n_train in SAMPLE_SIZES:
        if n_train > U_train_pool.shape[0]:
            raise ValueError(f"Requested N_train={n_train}, but train pool has only {U_train_pool.shape[0]}")
        bundle = train_models_for_sample_size(n_train, U_train_pool, U_val)
        trained_bundles.append(bundle)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Evaluate all sample sizes on the same common test set.
    rows = []
    for bundle in trained_bundles:
        row = evaluate_bundle_on_same_test_set(bundle, U_test, test_idx)
        rows.append(row)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    save_learning_curve_results(rows)

    print("\nDone.")
    print("Results directory:", RESULTS_DIR)
    print("Plots directory  :", PLOT_DIR)
    print("Models directory :", MODEL_DIR)


if __name__ == "__main__":
    main()

DATA_FILE = /content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
exists    = True
OUTPUT    = /content/drive/MyDrive/anus/ch2d_outputs
DATA_FILE = /content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
exists    = True
OUTPUT    = /content/drive/MyDrive/anus/ch2d_outputs
Using device: cuda

AUDIT FLAGS
-----------
DATA_FILE                         : /content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
OUTPUT_ROOT                       : /content/drive/MyDrive/anus/ch2d_outputs
CACHE_ROOT                        : /content/drive/MyDrive/anus/ch2d_outputs/cache_CH2D_TB_AWFNO_AWFNO_WFNO_FNO_EXACT_LKL_sizes25-50-100-200-400_K5_M4_W32_mx16_my16_R2_mean1
FORCE_RETRAIN                     : False
RUN_EXACT_HASH_OVERLAP_CHECK      : True
RUN_DUPLICATE_REPORT_INSIDE_SPLITS: True
RUN_ROLLOUT_INPUT_GAP_CHECK       : True
DEBUG_SHUFFLE_TRAIN_OUTPUTS       : False
DEBUG_RANDOM_BASIS                : False
MAKE_FIELD_PLOTS                  : True
SAVE_ROLLOUT_ARRAYS          

In [ ]:
DATA_FILE = Path("/content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt")
OUTPUT_ROOT = Path("/content/gdrive/MyDrive/anus/ch2d_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
!ls -lh /content/gdrive/MyDrive/anus/
print("DATA_FILE =", DATA_FILE)
print("exists    =", DATA_FILE.exists())
print("OUTPUT    =", OUTPUT_ROOT)

total 4.0K
drwxr-xr-x 3 root root 4.0K May  7 00:00 ch2d_outputs
DATA_FILE = /content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
exists    = False
OUTPUT    = /content/gdrive/MyDrive/anus/ch2d_outputs


In [ ]:
!find /content/gdrive/MyDrive/anus -type f | grep -i "cahn"